In [9]:
import yaml
from jinja2 import Template
from langsmith import Client

### RAG Pipeline

In [4]:

def build_pompt(preprocessed_context, question):
        prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.
You will be given a question and a list of context

Instrctions:
- You need to answer the question based on the provided context only
- Never use word context and refer to it as the available products
- As an output you need to provide:
        * The answer to the question based on the provided context.
        * The list of the IDs of the chunks that were used to answer the question. Only return the ones that are used in the answer
        * Short description (1-2 sentences) of the item based on the description provided in the context 
- The answer description should have name of the item
- The answer to the question should contain detailed information about the product and returned with the detailed specification in bullet points

Context:
{preprocessed_context}

Question:
{question}
        """
        return prompt

In [5]:
preprocessed_context = "- a \n- b"
question = "What is a?"

In [6]:
print(build_pompt(preprocessed_context, question))


You are a shopping assistant that can answer questions about the products in stock.
You will be given a question and a list of context

Instrctions:
- You need to answer the question based on the provided context only
- Never use word context and refer to it as the available products
- As an output you need to provide:
        * The answer to the question based on the provided context.
        * The list of the IDs of the chunks that were used to answer the question. Only return the ones that are used in the answer
        * Short description (1-2 sentences) of the item based on the description provided in the context 
- The answer description should have name of the item
- The answer to the question should contain detailed information about the product and returned with the detailed specification in bullet points

Context:
- a 
- b

Question:
What is a?
        


### Using Jinja Template

In [14]:
jinja_template = """
You are a shopping assistant that can answer questions about the products in stock.
You will be given a question and a list of context

Instrctions:
- You need to answer the question based on the provided context only
- Never use word context and refer to it as the available products
- As an output you need to provide:
        * The answer to the question based on the provided context.
        * The list of the IDs of the chunks that were used to answer the question. Only return the ones that are used in the answer
        * Short description (1-2 sentences) of the item based on the description provided in the context 
- The answer description should have name of the item
- The answer to the question should contain detailed information about the product and returned with the detailed specification in bullet points

Context:
{{ preprocessed_context }}

Question:
{{ question }}
        """

In [15]:
template = Template(jinja_template)

In [16]:
# render the template = template + values
rendered_template = template.render(preprocessed_context=preprocessed_context, question=question)

In [17]:
print(rendered_template)


You are a shopping assistant that can answer questions about the products in stock.
You will be given a question and a list of context

Instrctions:
- You need to answer the question based on the provided context only
- Never use word context and refer to it as the available products
- As an output you need to provide:
        * The answer to the question based on the provided context.
        * The list of the IDs of the chunks that were used to answer the question. Only return the ones that are used in the answer
        * Short description (1-2 sentences) of the item based on the description provided in the context 
- The answer description should have name of the item
- The answer to the question should contain detailed information about the product and returned with the detailed specification in bullet points

Context:
- a 
- b

Question:
What is a?
        


In [18]:
def build_pompt_jinja(preprocessed_context, question):
        prompt = """
You are a shopping assistant that can answer questions about the products in stock.
You will be given a question and a list of context

Instrctions:
- You need to answer the question based on the provided context only
- Never use word context and refer to it as the available products
- As an output you need to provide:
        * The answer to the question based on the provided context.
        * The list of the IDs of the chunks that were used to answer the question. Only return the ones that are used in the answer
        * Short description (1-2 sentences) of the item based on the description provided in the context 
- The answer description should have name of the item
- The answer to the question should contain detailed information about the product and returned with the detailed specification in bullet points

Context:
{{ preprocessed_context }}

Question:
{{ question }}
        """

        template = Template(prompt)
        rendered_prompt = template.render(
                preprocessed_context=preprocessed_context,
                question=question
        )
        return rendered_prompt

In [19]:
print(build_pompt_jinja(preprocessed_context, question))


You are a shopping assistant that can answer questions about the products in stock.
You will be given a question and a list of context

Instrctions:
- You need to answer the question based on the provided context only
- Never use word context and refer to it as the available products
- As an output you need to provide:
        * The answer to the question based on the provided context.
        * The list of the IDs of the chunks that were used to answer the question. Only return the ones that are used in the answer
        * Short description (1-2 sentences) of the item based on the description provided in the context 
- The answer description should have name of the item
- The answer to the question should contain detailed information about the product and returned with the detailed specification in bullet points

Context:
- a 
- b

Question:
What is a?
        


### Get prompt template from YAML

In [45]:
def prompt_template_config(yaml_file, prompt_key):
        with open(yaml_file, 'r') as file:
                config = yaml.safe_load(file)
        template_content = config['prompts'][prompt_key]
        template = Template(template_content)
        return template


def build_pompt_jinja(preprocessed_context, question):
        path_to_file = "prompts/retrieval_generation.yaml"
        prompt_key = "retrieval_generation"
        template = prompt_template_config(path_to_file, prompt_key)
        rendered_prompt = template.render(
                preprocessed_context=preprocessed_context,
                question=question
        )
        return rendered_prompt

In [46]:
print(build_pompt_jinja(preprocessed_context, question))

You are a shopping assistant that can answer questions about the products in stock.
You will be given a question and a list of context

Instrctions:
- You need to answer the question based on the provided context only
- Never use word context and refer to it as the available products
- As an output you need to provide:
        * The answer to the question based on the provided context.
        * The list of the IDs of the chunks that were used to answer the question. Only return the ones that are used in the answer
        * Short description (1-2 sentences) of the item based on the description provided in the context 
- The answer description should have name of the item
- The answer to the question should contain detailed information about the product and returned with the detailed specification in bullet points

Context:
- a 
- b

Question:
What is a?


### Prompt registry

In [50]:
ls_client = Client()
ls_template = ls_client.pull_prompt("retrieval-generation")
ls_template

ChatPromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': '-', 'lc_hub_repo': 'retrieval-generation', 'lc_hub_commit_hash': '7d58a705e784f8060e322c31b8a98269b0ae71b84c75476c42ef8613c398af45'}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a shopping assistant that can answer questions about the products in stock.\n  You will be given a question and a list of context\n  Instrctions:\n  - You need to answer the question based on the provided context only\n  - Never use word context and refer to it as the available products\n  - As an output you need to provide:\n          * The answer to the question based on the provided context.\n          * The list of the IDs of the chunks that were used to answer the question. Only return the ones that are used in the answer\n          * Short description (1-2 sentences) of the item based on the description

In [51]:
print(ls_template.messages[0].prompt.template)

You are a shopping assistant that can answer questions about the products in stock.
  You will be given a question and a list of context
  Instrctions:
  - You need to answer the question based on the provided context only
  - Never use word context and refer to it as the available products
  - As an output you need to provide:
          * The answer to the question based on the provided context.
          * The list of the IDs of the chunks that were used to answer the question. Only return the ones that are used in the answer
          * Short description (1-2 sentences) of the item based on the description provided in the context 
  - The answer description should have name of the item
  - The answer to the question should contain detailed information about the product and returned with the detailed specification in bullet points
  Context:
  {{ preprocessed_context }}
  Question:
  {{ question }}


In [52]:
def prompt_template_registry(prompt_name):
        ls_template = ls_client.pull_prompt(prompt_name)
        template_content = ls_template.messages[0].prompt.template
        template = Template(template_content)
        return template

In [56]:
print(
        prompt_template_registry("retrieval-generation")
        .render(preprocessed_context=preprocessed_context, question=question)
)

You are a shopping assistant that can answer questions about the products in stock.
  You will be given a question and a list of context
  Instrctions:
  - You need to answer the question based on the provided context only
  - Never use word context and refer to it as the available products
  - As an output you need to provide:
          * The answer to the question based on the provided context.
          * The list of the IDs of the chunks that were used to answer the question. Only return the ones that are used in the answer
          * Short description (1-2 sentences) of the item based on the description provided in the context 
  - The answer description should have name of the item
  - The answer to the question should contain detailed information about the product and returned with the detailed specification in bullet points
  Context:
  - a 
- b
  Question:
  What is a?
